In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Загрузка очищенного датасета
df = pd.read_pickle('D:\учеба\9 сем\mohov\lr1\iis\eda\heart_clean.pkl')

# Разделение на обучающую и тестовую выборки (75%-25%)
X = df.drop('target', axis=1) 
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (226, 13)
Размер тестовой выборки: (76, 13)


In [5]:
# Определение числовых и категориальных признаков
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Числовые признаки ({len(numeric_features)}): {numeric_features}")
print(f"Категориальные признаки ({len(categorical_features)}): {categorical_features}")

Числовые признаки (5): ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Категориальные признаки (8): ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestClassifier

# Создание трансформера для признаков
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', TargetEncoder(), categorical_features)
])

# Создание Pipeline
pipeline = Pipeline([
    ('transform', preprocessor),
    ('classification', RandomForestClassifier(random_state=42))
])

print("Pipeline создан")

Pipeline создан


In [7]:
# Обучение baseline-модели
pipeline.fit(X_train, y_train)

# Предсказания
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)

from sklearn.metrics import accuracy_score, classification_report
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7895

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.66      0.74        35
           1       0.76      0.90      0.82        41

    accuracy                           0.79        76
   macro avg       0.80      0.78      0.78        76
weighted avg       0.80      0.79      0.79        76



In [11]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import os

# Подготовка артефактов
input_example = X_train.head(5)
signature = infer_signature(model_input=X_train.head(5), model_output=pipeline.predict(X_train.head(5)))

# Логирование baseline модели
mlflow.set_experiment("Heart Disease Prediction")

with mlflow.start_run(run_name="Baseline_RandomForest") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "test_size": 0.25,
        "random_state": 42,
        "preprocessing": "StandardScaler + TargetEncoder"
    })
    
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )
    
    requirements_content = """numpy==1.26.4
        scikit-learn
        pandas
        mlflow==2.16"""
    
    with open("model_requirements.txt", "w") as f:
        f.write(requirements_content)
    
    mlflow.log_artifact("model_requirements.txt")
    os.remove("model_requirements.txt")  # Удаляем временный файл
    
    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/14 21:25:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run Baseline_RandomForest at: http://127.0.0.1:5000/#/experiments/1/runs/332d724952b24c84a96fd0e22ce90d09.
2025/10/14 21:25:12 INFO mlflow.tracking._tracking_service.cli

Run ID: 332d724952b24c84a96fd0e22ce90d09
Accuracy: 0.7895


In [14]:
# Регистрация модели
model_uri = f"runs:/{run.info.run_id}/model"
model_name = "heart_disease_model"

mlflow.register_model(model_uri, model_name)
print(f"Модель зарегистрирована как {model_name}")

Successfully registered model 'heart_disease_model'.
2025/10/14 21:25:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart_disease_model, version 1


Модель зарегистрирована как heart_disease_model


Created version '1' of model 'heart_disease_model'.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import pickle

# Создаем копию обучающих данных
X_train_fe_sklearn = X_train.copy()

poly_features = numeric_features[:3]


bins_features =  numeric_features[-2:]

print(f"Признаки для PolynomialFeatures: {poly_features}")
print(f"Признаки для KBinsDiscretizer: {bins_features}")

# Создаем ColumnTransformer с расширенными трансформациями
preprocessor_extended = ColumnTransformer([
    # Базовые трансформации
    ('num_standard', StandardScaler(), numeric_features),
    ('cat_ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
    
    # Новые трансформации
    ('poly', Pipeline([
        ('poly_feat', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler())
    ]), poly_features),
    ('bins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform'), bins_features)
])

# Создаем полный pipeline
pipeline_extended = Pipeline([
    ('transform', preprocessor_extended),
    ('classification', RandomForestClassifier(random_state=42))
])

pipeline_extended.fit(X_train, y_train)

# Получаем трансформированные данные
X_train_fe_sklearn = pipeline_extended.named_steps['transform'].transform(X_train)

feature_names = []

# Стандартные числовые признаки
feature_names.extend([f"num_standard__{col}" for col in numeric_features])

# Категориальные признаки
feature_names.extend([f"cat_ordinal__{col}" for col in categorical_features])

# Полиномиальные признаки
poly_transformer = PolynomialFeatures(degree=2, include_bias=False)
poly_transformer.fit(X_train[poly_features])
poly_names = [f"poly__{name}" for name in poly_transformer.get_feature_names_out(poly_features)]
feature_names.extend(poly_names)

# Биннированные признаки
feature_names.extend([f"bins__{col}" for col in bins_features])

print(f"Общее количество признаков: {len(feature_names)}")

with open('feature_names_sklearn.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

y_pred_extended = pipeline_extended.predict(X_test)
accuracy_extended = accuracy_score(y_test, y_pred_extended)

print(f"Accuracy с расширенными признаками: {accuracy_extended:.4f}")

Признаки для PolynomialFeatures: ['age', 'trestbps', 'chol']
Признаки для KBinsDiscretizer: ['chol', 'thalach', 'oldpeak']
Общее количество признаков: 25
Accuracy с расширенными признаками: 0.7105


In [ ]:
with mlflow.start_run(run_name="Extended_Features_sklearn") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier", 
        "feature_engineering": "PolynomialFeatures + KBinsDiscretizer",
        "poly_degree": 2,
        "n_bins": 5,
        "total_features": len(feature_names)
    })
    
    mlflow.log_metric("accuracy", accuracy_extended)
    
    # Подготовка входного примера и сигнатуры
    input_example_extended = X_train.head(5)
    signature_extended = infer_signature(
        model_input=input_example_extended,
        model_output=pipeline_extended.predict(input_example_extended)
    )
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_extended,
        artifact_path="model",
        signature=signature_extended,
        input_example=input_example_extended
    )
    
    mlflow.log_artifact('feature_names_sklearn.pkl')
    
    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {accuracy_extended:.4f}")

import os
os.remove('feature_names_sklearn.pkl')

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/14 21:32:29 INFO mlflow.tracking._tracking_service.client: 🏃 View run Extended_Features_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/d320867c7e6941deb4aa3e6d855b1aab.
2025/10/14 21:32:29 INFO mlflow.tracking._tracking_service

Run ID: d320867c7e6941deb4aa3e6d855b1aab
Accuracy: 0.7105


In [ ]:
from autofeat import AutoFeatClassifier
import pandas as pd

# Создаем копию данных для autofeat
X_train_autofeat = X_train.copy()

# autofeat работает только с числовыми данными, поэтому предобработаем категориальные
if len(categorical_features) > 0:
    from sklearn.preprocessing import LabelEncoder
    X_train_autofeat_processed = X_train_autofeat.copy()
    
    for col in categorical_features:
        le = LabelEncoder()
        X_train_autofeat_processed[col] = le.fit_transform(X_train_autofeat_processed[col].astype(str))
else:
    X_train_autofeat_processed = X_train_autofeat.copy()

# Создаем AutoFeat трансформер
autofeat_transformer = AutoFeatClassifier(
    feateng_steps=2,
    max_gb=1,
    verbose=1
)

# Обучаем autofeat
X_train_autofeat_transformed = autofeat_transformer.fit_transform(X_train_autofeat_processed, y_train)

print(f"Исходное количество признаков: {X_train_autofeat_processed.shape[1]}")
print(f"После autofeat: {X_train_autofeat_transformed.shape[1]}")

# Создаем pipeline с autofeat
pipeline_autofeat = Pipeline([
    ('autofeat', autofeat_transformer),
    ('classification', RandomForestClassifier(random_state=42))
])

# Предобработка тестовых данных аналогично обучающим
X_test_autofeat = X_test.copy()
if len(categorical_features) > 0:
    for col in categorical_features:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_test[col]]).astype(str)
        le.fit(combined)
        X_test_autofeat[col] = le.transform(X_test[col].astype(str))

# Предсказания
pipeline_autofeat.fit(X_train_autofeat_processed, y_train)
y_pred_autofeat = pipeline_autofeat.predict(X_test_autofeat)
accuracy_autofeat = accuracy_score(y_test, y_pred_autofeat)

print(f"Accuracy с autofeat: {accuracy_autofeat:.4f}")


2025-10-14 21:42:22,024 INFO: [AutoFeat] The 2 step feature engineering process could generate up to 4186 features.
2025-10-14 21:42:22,025 INFO: [AutoFeat] With 226 data points this new feature matrix would use about 0.00 gb of space.
2025-10-14 21:42:22,026 INFO: [feateng] Step 1: transformation of original features


2025-10-14 21:42:23,057 INFO: [feateng] Generated 44 transformed features from 13 original features - done.
2025-10-14 21:42:23,057 INFO: [feateng] Step 2: first combination of features


2025-10-14 21:42:23,659 INFO: [feateng] Generated 1572 feature combinations from 1596 original feature tuples - done.
2025-10-14 21:42:23,662 INFO: [feateng] Generated altogether 1625 new features in 2 steps
2025-10-14 21:42:23,663 INFO: [feateng] Removing correlated features, as well as additions at the highest level
2025-10-14 21:42:23,674 INFO: [feateng] Generated a total of 870 additional features
2025-10-14 21:42:23,678 INFO: [featsel] Feature selection run 1/5


[featsel] Scaling data...done.       1596 feature tuples combined


2025-10-14 21:42:31,752 INFO: [featsel] Feature selection run 2/5
2025-10-14 21:42:40,599 INFO: [featsel] Feature selection run 3/5
2025-10-14 21:42:48,104 INFO: [featsel] Feature selection run 4/5
2025-10-14 21:42:55,749 INFO: [featsel] Feature selection run 5/5
2025-10-14 21:43:03,643 INFO: [featsel] 7 features after 5 feature selection runs
d:\учеба\9 сем\mohov\.venv\Lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
2025-10-14 21:43:03,643 INFO: [featsel] 7 features after correlation filtering
2025-10-14 21:43:03,695 INFO: [featsel] 5 features after noise filtering
2025-10-14 21:43:03,695 INFO: [AutoFeat] Computing 5 new features.


...

2025-10-14 21:43:13,260 INFO: [featsel] Feature selection run 2/5
2025-10-14 21:43:21,697 INFO: [featsel] Feature selection run 3/5
2025-10-14 21:43:29,144 INFO: [featsel] Feature selection run 4/5
2025-10-14 21:43:36,650 INFO: [featsel] Feature selection run 5/5
2025-10-14 21:43:44,422 INFO: [featsel] 7 features after 5 feature selection runs
d:\учеба\9 сем\mohov\.venv\Lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
2025-10-14 21:43:44,427 INFO: [featsel] 7 features after correlation filtering
2025-10-14 21:43:44,472 INFO: [featsel] 5 features after noise filtering
2025-10-14 21:43:44,472 INFO: [AutoFeat] Computing 5 new features.


2025-10-14 21:43:45,152 INFO: [AutoFeat]     5/    5 new features ...done.
2025-10-14 21:43:45,153 INFO: [AutoFeat] Final dataframe with 18 feature columns (5 new).
2025-10-14 21:43:45,154 INFO: [AutoFeat] Training final classification model.


2025-10-14 21:43:45,305 INFO: [AutoFeat] Trained model: largest coefficients:
2025-10-14 21:43:45,305 INFO: [0.9806501]
2025-10-14 21:43:45,305 INFO: 0.677457 * sqrt(oldpeak)*sex
2025-10-14 21:43:45,305 INFO: 0.625932 * sqrt(ca)*sex
2025-10-14 21:43:45,305 INFO: 0.000412 * chol*exp(thal)
2025-10-14 21:43:45,305 INFO: [AutoFeat] Final score: 0.8540
d:\учеба\9 сем\mohov\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2025-10-14 21:43:45,376 INFO: [AutoFeat] Computing 5 new features.
2025-10-14 21:43:45,379 INFO: [AutoFeat]     5/    5 new features ...done.


Accuracy с autofeat: 0.7895features


AttributeError: 'AutoFeatClassifier' object has no attribute 'cols_out_'

In [ ]:
autofeat_features = list(X_train_autofeat_transformed.columns)

with open('autofeat_features.pkl', 'wb') as f:
    pickle.dump(autofeat_features, f)

print(f"Сохранено {len(autofeat_features)} признаков после autofeat.")


Сохранено 18 признаков после autofeat.


In [ ]:
with mlflow.start_run(run_name="AutoFeat_Features") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "feature_engineering": "AutoFeat",
        "feateng_steps": 2,
        "original_features": X_train_autofeat_processed.shape[1],
        "generated_features": X_train_autofeat_transformed.shape[1]
    })
    
    mlflow.log_metric("accuracy", accuracy_autofeat)
    
    # Входной пример для autofeat
    input_example_autofeat = X_train_autofeat_processed.head(5)
    signature_autofeat = infer_signature(
        model_input=input_example_autofeat,
        model_output=pipeline_autofeat.predict(input_example_autofeat)
    )
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_autofeat,
        artifact_path="model",
        signature=signature_autofeat,
        input_example=input_example_autofeat
    )
    
    mlflow.log_artifact('autofeat_features.pkl')
    
    print(f"AutoFeat Run ID: {run.info.run_id}")

os.remove('autofeat_features.pkl')

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2025-10-14 21:44:51,649 INFO: [AutoFeat] Computing 5 new features.
2025-10-14 21:44:51,652 INFO: [AutoFeat]     5/    5 new features ...done.
d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/d

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2025-10-14 21:44:54,484 INFO: [AutoFeat] Computing 5 new features.


2025-10-14 21:44:55,037 INFO: [AutoFeat]     5/    5 new features ...done.
2025/10/14 21:44:55 INFO mlflow.tracking._tracking_service.client: 🏃 View run AutoFeat_Features at: http://127.0.0.1:5000/#/experiments/1/runs/c7b6a21e2c744cb2859025c508c714eb.
2025/10/14 21:44:55 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


AutoFeat Run ID: c7b6a21e2c744cb2859025c508c714eb


In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.metrics import accuracy_score
import numpy as np

total_features = X_train_fe_sklearn.shape[1]
n_features_to_select = int(total_features * 0.4) 

print(f"Общее количество признаков: {total_features}")
print(f"Будем отбирать: {n_features_to_select}")

# Создаем базовую модель для отбора признаков
rf_for_selection = RandomForestClassifier(n_estimators=50, random_state=42)

# Создаем SequentialFeatureSelector с forward selection
sfs = SFS(
    rf_for_selection,
    k_features=n_features_to_select,
    forward=True,
    floating=False,
    verbose=2,
    scoring='accuracy',
    cv=3
)

sfs.fit(X_train_fe_sklearn, y_train)

# Получаем индексы отобранных признаков
selected_indices = list(sfs.k_feature_idx_)
selected_names = [feature_names[i] for i in selected_indices]

print(f"\nОтобранные признаки ({len(selected_indices)}):")
for i, name in enumerate(selected_names):
    print(f"{i+1}: {name}")

# Сохраняем информацию об отобранных признаках
with open('selected_feature_indices.pkl', 'wb') as f:
    pickle.dump(selected_indices, f)
    
with open('selected_feature_names.pkl', 'wb') as f:
    pickle.dump(selected_names, f)

from sklearn.feature_selection import SelectFromModel

class IndexSelector:
    def __init__(self, indices):
        self.indices = indices
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X[:, self.indices]
    
    def fit_transform(self, X, y=None):
        return self.transform(X)

# Создаем полный pipeline с отбором признаков
pipeline_with_selection = Pipeline([
    ('transform', preprocessor_extended),  # Из пункта 10
    ('feature_selection', IndexSelector(selected_indices)),
    ('classification', RandomForestClassifier(random_state=42))
])

pipeline_with_selection.fit(X_train, y_train)

y_pred_selected = pipeline_with_selection.predict(X_test)
accuracy_selected = accuracy_score(y_test, y_pred_selected)

print(f"\nAccuracy с отбором признаков: {accuracy_selected:.4f}")
print(f"Было признаков: {total_features}, стало: {len(selected_indices)}")

Общее количество признаков: 25
Будем отбирать: 10


[Parallel(n_jobs=1)]: Done  25 out of  25 | elapsed:    2.4s finished

[2025-10-14 21:45:47] Features: 1/10 -- score: 0.7922222222222222[Parallel(n_jobs=1)]: Done  24 out of  24 | elapsed:    2.2s finished

[2025-10-14 21:45:49] Features: 2/10 -- score: 0.8009941520467837[Parallel(n_jobs=1)]: Done  23 out of  23 | elapsed:    2.2s finished

[2025-10-14 21:45:51] Features: 3/10 -- score: 0.8363157894736842[Parallel(n_jobs=1)]: Done  22 out of  22 | elapsed:    2.1s finished

[2025-10-14 21:45:53] Features: 4/10 -- score: 0.832046783625731[Parallel(n_jobs=1)]: Done  21 out of  21 | elapsed:    2.1s finished

[2025-10-14 21:45:55] Features: 5/10 -- score: 0.8673099415204678[Parallel(n_jobs=1)]: Done  20 out of  20 | elapsed:    2.0s finished

[2025-10-14 21:45:58] Features: 6/10 -- score: 0.8542690058479532[Parallel(n_jobs=1)]: Done  19 out of  19 | elapsed:    1.9s finished

[2025-10-14 21:46:00] Features: 7/10 -- score: 0.8629239766081871[Parallel(n_jobs=1)]: Done  18 out of  18 | elaps


Отобранные признаки (10):
1: cat_ordinal__sex
2: cat_ordinal__cp
3: cat_ordinal__fbs
4: cat_ordinal__restecg
5: cat_ordinal__slope
6: cat_ordinal__ca
7: cat_ordinal__thal
8: poly__chol
9: poly__trestbps^2
10: bins__oldpeak

Accuracy с отбором признаков: 0.7895
Было признаков: 25, стало: 10


[Parallel(n_jobs=1)]: Done  16 out of  16 | elapsed:    1.6s finished

[2025-10-14 21:46:05] Features: 10/10 -- score: 0.8452631578947368

In [ ]:
with mlflow.start_run(run_name="Feature_Selection_Forward") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "feature_engineering": "PolynomialFeatures + KBinsDiscretizer + Forward_SFS",
        "selection_method": "Sequential_Forward_Selection",
        "original_features": total_features,
        "selected_features": len(selected_indices),
        "selection_ratio": len(selected_indices) / total_features,
        "cv_folds": 3
    })
    
    mlflow.log_metric("accuracy", accuracy_selected)
    
    # Входной пример
    input_example_selected = X_train.head(5)
    signature_selected = infer_signature(
        model_input=input_example_selected,
        model_output=pipeline_with_selection.predict(input_example_selected)
    )
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_with_selection,
        artifact_path="model",
        signature=signature_selected,
        input_example=input_example_selected
    )
    
    mlflow.log_artifact('selected_feature_indices.pkl')
    mlflow.log_artifact('selected_feature_names.pkl')
    
    print(f"Feature Selection Run ID: {run.info.run_id}")

os.remove('selected_feature_indices.pkl')
os.remove('selected_feature_names.pkl')

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/14 21:46:26 INFO mlflow.tracking._tracking_service.client: 🏃 View run Feature_Selection_Forward at: http://127.0.0.1:5000/#/experiments/1/runs/3cb476b78f0e425589a3a8b4379ecfcc.
2025/10/14 21:46:26 INFO mlflow.tracking._tracking_service

Feature Selection Run ID: 3cb476b78f0e425589a3a8b4379ecfcc


In [ ]:
from sklearn.feature_selection import RFE

print("=== Sequential Backward Selection ===")

sfs_backward = SFS(
    RandomForestClassifier(n_estimators=50, random_state=42),
    k_features=n_features_to_select,
    forward=False,  # Backward selection
    floating=False,
    verbose=2,
    scoring='accuracy',
    cv=3
)

sfs_backward.fit(X_train_fe_sklearn, y_train)

selected_indices_backward = list(sfs_backward.k_feature_idx_)
selected_names_backward = [feature_names[i] for i in selected_indices_backward]

print(f"\nОтобранные признаки Backward SFS ({len(selected_indices_backward)}):")
for i, name in enumerate(selected_names_backward):
    print(f"{i+1}: {name}")

print("\n=== Recursive Feature Elimination ===")

rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    n_features_to_select=n_features_to_select,
    verbose=1
)

rfe.fit(X_train_fe_sklearn, y_train)

selected_indices_rfe = np.where(rfe.support_)[0].tolist()
selected_names_rfe = [feature_names[i] for i in selected_indices_rfe]

print(f"\nОтобранные признаки RFE ({len(selected_indices_rfe)}):")
for i, name in enumerate(selected_names_rfe):
    print(f"{i+1}: {name}")

forward_set = set(selected_indices)
backward_set = set(selected_indices_backward)
rfe_set = set(selected_indices_rfe)

# Пересечения
forward_backward = forward_set.intersection(backward_set)
forward_rfe = forward_set.intersection(rfe_set)
backward_rfe = backward_set.intersection(rfe_set)
all_three = forward_set.intersection(backward_set).intersection(rfe_set)

print(f"\n=== Анализ пересечений ===")
print(f"Forward ∩ Backward: {len(forward_backward)} признаков")
print(f"Forward ∩ RFE: {len(forward_rfe)} признаков")
print(f"Backward ∩ RFE: {len(backward_rfe)} признаков")
print(f"Все три метода: {len(all_three)} признаков")

if len(all_three) > 0:
    print(f"\nПризнаки, выбранные всеми тремя методами:")
    common_names = [feature_names[i] for i in all_three]
    for name in common_names:
        print(f"  - {name}")

# Объединение всех выбранных признаков
union_indices = list(forward_set.union(backward_set).union(rfe_set))
print(f"\nОбъединение всех методов: {len(union_indices)} признаков")

results = {
    'forward': selected_indices,
    'backward': selected_indices_backward,
    'rfe': selected_indices_rfe,
    'common_all': list(all_three),
    'union_all': union_indices
}

with open('feature_selection_comparison.pkl', 'wb') as f:
    pickle.dump(results, f)

=== Sequential Backward Selection ===


[Parallel(n_jobs=1)]: Done  25 out of  25 | elapsed:    2.7s finished

[2025-10-14 21:47:24] Features: 24/10 -- score: 0.8185380116959063[Parallel(n_jobs=1)]: Done  24 out of  24 | elapsed:    2.7s finished

[2025-10-14 21:47:27] Features: 23/10 -- score: 0.8187719298245614[Parallel(n_jobs=1)]: Done  23 out of  23 | elapsed:    2.5s finished

[2025-10-14 21:47:29] Features: 22/10 -- score: 0.8363157894736842[Parallel(n_jobs=1)]: Done  22 out of  22 | elapsed:    2.4s finished

[2025-10-14 21:47:32] Features: 21/10 -- score: 0.8098245614035088[Parallel(n_jobs=1)]: Done  21 out of  21 | elapsed:    2.2s finished

[2025-10-14 21:47:34] Features: 20/10 -- score: 0.831812865497076[Parallel(n_jobs=1)]: Done  20 out of  20 | elapsed:    2.1s finished

[2025-10-14 21:47:36] Features: 19/10 -- score: 0.840701754385965[Parallel(n_jobs=1)]: Done  19 out of  19 | elapsed:    1.9s finished

[2025-10-14 21:47:38] Features: 18/10 -- score: 0.840701754385965[Parallel(n_jobs=1)]: Done  18 out of  18 | 


Отобранные признаки Backward SFS (10):
1: num_standard__age
2: num_standard__oldpeak
3: cat_ordinal__sex
4: cat_ordinal__cp
5: cat_ordinal__slope
6: cat_ordinal__ca
7: cat_ordinal__thal
8: poly__chol
9: poly__age chol
10: poly__trestbps chol

=== Recursive Feature Elimination ===
Fitting estimator with 25 features.
Fitting estimator with 24 features.
Fitting estimator with 23 features.
Fitting estimator with 22 features.
Fitting estimator with 21 features.
Fitting estimator with 20 features.
Fitting estimator with 19 features.
Fitting estimator with 18 features.
Fitting estimator with 17 features.
Fitting estimator with 16 features.
Fitting estimator with 15 features.
Fitting estimator with 14 features.
Fitting estimator with 13 features.
Fitting estimator with 12 features.
Fitting estimator with 11 features.

Отобранные признаки RFE (10):
1: num_standard__age
2: num_standard__chol
3: num_standard__thalach
4: num_standard__oldpeak
5: cat_ordinal__cp
6: cat_ordinal__ca
7: cat_ordinal__

In [ ]:
pipeline_backward = Pipeline([
    ('transform', preprocessor_extended),
    ('feature_selection', IndexSelector(selected_indices_backward)),
    ('classification', RandomForestClassifier(random_state=42))
])

pipeline_backward.fit(X_train, y_train)
y_pred_backward = pipeline_backward.predict(X_test)
accuracy_backward = accuracy_score(y_test, y_pred_backward)

pipeline_rfe = Pipeline([
    ('transform', preprocessor_extended),
    ('feature_selection', IndexSelector(selected_indices_rfe)),
    ('classification', RandomForestClassifier(random_state=42))
])

pipeline_rfe.fit(X_train, y_train)
y_pred_rfe = pipeline_rfe.predict(X_test)
accuracy_rfe = accuracy_score(y_test, y_pred_rfe)


# 4. Модель с объединением всех методов
if len(union_indices) < total_features * 0.8: 
    pipeline_union = Pipeline([
        ('transform', preprocessor_extended),
        ('feature_selection', IndexSelector(union_indices)),
        ('classification', RandomForestClassifier(random_state=42))
    ])
    
    pipeline_union.fit(X_train, y_train)
    y_pred_union = pipeline_union.predict(X_test)
    accuracy_union = accuracy_score(y_test, y_pred_union)
else:
    accuracy_union = None

print(f"\n=== Сравнение результатов ===")
print(f"Forward Selection: {accuracy_selected:.4f}")
print(f"Backward Selection: {accuracy_backward:.4f}")
print(f"RFE: {accuracy_rfe:.4f}")

if accuracy_union:
    print(f"Объединение: {accuracy_union:.4f}")


=== Сравнение результатов ===
Forward Selection: 0.7895
Backward Selection: 0.7763
RFE: 0.7763
Объединение: 0.8026


In [ ]:
with mlflow.start_run(run_name="Feature_Selection_Backward") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "selection_method": "Sequential_Backward_Selection",
        "selected_features": len(selected_indices_backward)
    })
    mlflow.log_metric("accuracy", accuracy_backward)
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_backward,
        artifact_path="model",
        signature=signature_selected,
        input_example=input_example_selected
    )

with mlflow.start_run(run_name="Feature_Selection_RFE") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "selection_method": "Recursive_Feature_Elimination",
        "selected_features": len(selected_indices_rfe)
    })
    mlflow.log_metric("accuracy", accuracy_rfe)
    
    mlflow.sklearn.log_model(
        sk_model=pipeline_rfe,
        artifact_path="model",
        signature=signature_selected,
        input_example=input_example_selected
    )

with mlflow.start_run(run_name="Feature_Selection_Comparison") as run:
    mlflow.log_params({
        "forward_features": len(selected_indices),
        "backward_features": len(selected_indices_backward),
        "rfe_features": len(selected_indices_rfe),
        "common_all_methods": len(all_three),
        "union_all_methods": len(union_indices)
    })
    
    mlflow.log_metrics({
        "accuracy_forward": accuracy_selected,
        "accuracy_backward": accuracy_backward,
        "accuracy_rfe": accuracy_rfe
    })
    
    if accuracy_common:
        mlflow.log_metric("accuracy_common", accuracy_common)
    if accuracy_union:
        mlflow.log_metric("accuracy_union", accuracy_union)
    
    mlflow.log_artifact('feature_selection_comparison.pkl')

os.remove('feature_selection_comparison.pkl')

2025/10/14 21:48:44 INFO mlflow.tracking._tracking_service.client: 🏃 View run Feature_Selection_Backward at: http://127.0.0.1:5000/#/experiments/1/runs/818496fbed1b46ffaa24bafbb07a83f7.
2025/10/14 21:48:44 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
2025/10/14 21:48:46 INFO mlflow.tracking._tracking_service.client: 🏃 View run Feature_Selection_RFE at: http://127.0.0.1:5000/#/experiments/1/runs/68127dbe7d2a4d30879f97d77d2b6b21.
2025/10/14 21:48:46 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
2025/10/14 21:48:46 INFO mlflow.tracking._tracking_service.client: 🏃 View run Feature_Selection_Comparison at: http://127.0.0.1:5000/#/experiments/1/runs/aec08daa49b840319d15e4ec2120b728.
2025/10/14 21:48:46 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer
import numpy as np

# Определяем лучшую модель из предыдущих экспериментов: 0.7895
best_pipeline = pipeline_with_selection

# Определяем пространство гиперпараметров
param_distributions = {
    'classification__n_estimators': [50, 100, 200, 300, 500],
    'classification__max_depth': [None, 5, 10, 15, 20, 25],
    'classification__max_features': np.arange(0.1, 1.1, 0.1)  # от 0.1 до 1.0
}

# Создаем scorer для f1 (нужно максимизировать)
f1_scorer = make_scorer(f1_score, average='weighted')

random_search = RandomizedSearchCV(
    estimator=best_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    cv=3,
    scoring=f1_scorer,  # Максимизируем f1_score
    random_state=42,
    verbose=2,
    n_jobs=1
)

print("Начинаем подбор гиперпараметров...")
random_search.fit(X_train, y_train)

print(f"\nЛучшие параметры: {random_search.best_params_}")
print(f"Лучший f1_score (CV): {random_search.best_score_:.4f}")

# Предсказания на тесте с лучшей моделью
best_model = random_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
f1_tuned = f1_score(y_test, y_pred_tuned, average='weighted')
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"F1-score на тесте: {f1_tuned:.4f}")
print(f"Accuracy на тесте: {accuracy_tuned:.4f}")

with mlflow.start_run(run_name="RandomForest_Hyperparameter_Tuning") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "hyperparameter_tuning": "RandomizedSearchCV",
        "cv_folds": 3,
        "n_iter": 15,
        "optimization_metric": "f1_score_weighted",
        "optimization_direction": "maximize"
    })
    
    for param, value in random_search.best_params_.items():
        mlflow.log_param(f"best_{param}", value)
    
    mlflow.log_metrics({
        "best_cv_f1_score": random_search.best_score_,
        "test_f1_score": f1_tuned,
        "test_accuracy": accuracy_tuned
    })
    
    input_example_tuned = X_train.head(5)
    signature_tuned = infer_signature(
        model_input=input_example_tuned,
        model_output=best_model.predict(input_example_tuned)
    )
    
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature_tuned,
        input_example=input_example_tuned
    )
    
    tuned_run_id = run.info.run_id
    print(f"Tuned Model Run ID: {tuned_run_id}")

# Регистрируем как версию 2
model_uri_v2 = f"runs:/{tuned_run_id}/model"
mlflow.register_model(model_uri_v2, model_name, tags={"version": "2", "method": "hyperparameter_tuning"})
print("Модель зарегистрирована как версия 2")

Начинаем подбор гиперпараметров...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END classification__max_depth=20, classification__max_features=0.1, classification__n_estimators=300; total time=   0.1s
[CV] END classification__max_depth=20, classification__max_features=0.1, classification__n_estimators=300; total time=   0.1s
[CV] END classification__max_depth=20, classification__max_features=0.1, classification__n_estimators=300; total time=   0.1s
[CV] END classification__max_depth=25, classification__max_features=0.4, classification__n_estimators=100; total time=   0.0s
[CV] END classification__max_depth=25, classification__max_features=0.4, classification__n_estimators=100; total time=   0.0s
[CV] END classification__max_depth=25, classification__max_features=0.4, classification__n_estimators=100; total time=   0.0s
[CV] END classification__max_depth=15, classification__max_features=0.1, classification__n_estimators=200; total time=   0.0s
[CV] END classificatio

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/14 21:52:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run RandomForest_Hyperparameter_Tuning at: http://127.0.0.1:5000/#/experiments/1/runs/c6e24d7d17e54270bf6bf651cc29c7d7.
2025/10/14 21:52:48 INFO mlflow.tracking._trackin

Tuned Model Run ID: c6e24d7d17e54270bf6bf651cc29c7d7
Модель зарегистрирована как версия 2


Created version '2' of model 'heart_disease_model'.


In [ ]:


from catboost import CatBoostClassifier
import mlflow.catboost

X_train_cat = X_train.copy()
X_test_cat = X_test.copy()

# Определяем категориальные признаки для CatBoost
cat_features_indices = [X_train_cat.columns.get_loc(col) for col in categorical_features if col in X_train_cat.columns]

catboost_params = {
    'iterations': [100, 200, 300],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7]
}

catboost_model = CatBoostClassifier(
    cat_features=cat_features_indices,
    random_seed=42,
    verbose=False
)

catboost_search = RandomizedSearchCV(
    estimator=catboost_model,
    param_distributions=catboost_params,
    n_iter=12,
    cv=3,
    scoring=f1_scorer,  # Максимизируем f1_score
    random_state=42,
    verbose=2
)

print("Начинаем подбор гиперпараметров для CatBoost...")
catboost_search.fit(X_train_cat, y_train)

best_catboost = catboost_search.best_estimator_
y_pred_catboost = best_catboost.predict(X_test_cat)
f1_catboost = f1_score(y_test, y_pred_catboost, average='weighted')
accuracy_catboost = accuracy_score(y_test, y_pred_catboost)

print(f"\nCatBoost лучшие параметры: {catboost_search.best_params_}")
print(f"CatBoost F1-score: {f1_catboost:.4f}")
print(f"CatBoost Accuracy: {accuracy_catboost:.4f}")

with mlflow.start_run(run_name="CatBoost_Hyperparameter_Tuning") as run:
    mlflow.log_params({
        "model_type": "CatBoostClassifier",
        "hyperparameter_tuning": "RandomizedSearchCV",
        "cv_folds": 3,
        "n_iter": 12,
        "optimization_metric": "f1_score_weighted",
        "optimization_direction": "maximize",
        "categorical_features": len(cat_features_indices)
    })
    
    for param, value in catboost_search.best_params_.items():
        mlflow.log_param(f"best_{param}", value)
    
    mlflow.log_metrics({
        "best_cv_f1_score": catboost_search.best_score_,
        "test_f1_score": f1_catboost,
        "test_accuracy": accuracy_catboost
    })
    
    mlflow.catboost.log_model(
        cb_model=best_catboost,
        artifact_path="model",
        signature=infer_signature(X_train_cat.head(5), best_catboost.predict(X_train_cat.head(5))),
        input_example=X_train_cat.head(5)
    )
    
    catboost_run_id = run.info.run_id
    print(f"CatBoost Run ID: {catboost_run_id}")

# Регистрируем как версию 3
model_uri_v3 = f"runs:/{catboost_run_id}/model"
mlflow.register_model(model_uri_v3, model_name, tags={"version": "3", "method": "catboost"})
print("CatBoost модель зарегистрирована как версия 3")

Начинаем подбор гиперпараметров для CatBoost...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END depth=10, iterations=100, l2_leaf_reg=7, learning_rate=0.01; total time=   3.4s
[CV] END depth=10, iterations=100, l2_leaf_reg=7, learning_rate=0.01; total time=   3.1s
[CV] END depth=10, iterations=100, l2_leaf_reg=7, learning_rate=0.01; total time=   4.1s
[CV] END depth=4, iterations=200, l2_leaf_reg=5, learning_rate=0.1; total time=   3.7s
[CV] END depth=4, iterations=200, l2_leaf_reg=5, learning_rate=0.1; total time=   3.7s
[CV] END depth=4, iterations=200, l2_leaf_reg=5, learning_rate=0.1; total time=   3.4s
[CV] END depth=8, iterations=100, l2_leaf_reg=7, learning_rate=0.1; total time=   3.0s
[CV] END depth=8, iterations=100, l2_leaf_reg=7, learning_rate=0.1; total time=   3.2s
[CV] END depth=8, iterations=100, l2_leaf_reg=7, learning_rate=0.1; total time=   3.4s
[CV] END depth=8, iterations=300, l2_leaf_reg=1, learning_rate=0.1; total time=  10.9s
[CV] END depth=

d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/10/14 21:57:29 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost_Hyperparameter_Tuning at: http://127.0.0.1:5000/#/experiments/1/runs/e14f2ac7b8a140cca935386d33b2e44d.
2025/10/14 21:57:29 INFO mlflow.tracking._tracking_se

CatBoost Run ID: e14f2ac7b8a140cca935386d33b2e44d
CatBoost модель зарегистрирована как версия 3


Created version '3' of model 'heart_disease_model'.


In [ ]:

best_final_model = Pipeline([
    ('transform', preprocessor_extended),  # Из пункта 10
    ('feature_selection', IndexSelector(selected_indices)),  # Из пункта 12
    ('classification', RandomForestClassifier(
        n_estimators=200,
        max_features=0.1,
        max_depth=15,
        random_state=42
    ))
])

# Объединяем train и test для обучения на всей выборке
X_full = pd.concat([X_train, X_test], axis=0)
y_full = pd.concat([y_train, y_test], axis=0)

print(f"Размер полной выборки: {X_full.shape}")

best_final_model.fit(X_full, y_full)

input_example_prod = X_full.head(5)
signature_prod = infer_signature(
    model_input=input_example_prod,
    model_output=best_final_model.predict(input_example_prod)
)

used_columns = list(X_full.columns)
with open('production_columns.pkl', 'wb') as f:
    pickle.dump(used_columns, f)

print(f"Используемые столбцы ({len(used_columns)}): {used_columns}")

with mlflow.start_run(run_name="Production_Model_Full_Dataset") as run:
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "training_data": "full_dataset",
        "n_estimators": 200,
        "max_features": 0.1,
        "max_depth": 15,
        "feature_engineering": "PolynomialFeatures + KBinsDiscretizer + Forward_SFS",
        "selected_features": len(selected_indices),
        "total_samples": len(X_full),
        "features_count": len(used_columns)
    })
    
    mlflow.sklearn.log_model(
        sk_model=best_final_model,
        artifact_path="model",
        signature=signature_prod,
        input_example=input_example_prod
    )
    
    mlflow.log_artifact('production_columns.pkl')
    mlflow.log_artifact('../requirements.txt')
    
    production_run_id = run.info.run_id
    print(f"Production Model Run ID: {production_run_id}")

model_uri_prod = f"runs:/{production_run_id}/model"

model_version = mlflow.register_model(model_uri_prod, model_name)
version_number = model_version.version

from mlflow.tracking import MlflowClient
client = MlflowClient()

client.set_model_version_tag(
    name=model_name,
    version=version_number,
    key="stage",
    value="Production"
)

print(f"Production модель зарегистрирована как версия {version_number} с тегом 'Production'")
print(f"Production Run ID: {production_run_id}")

os.remove('production_columns.pkl')

print(f"\n=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ ===")
print(f"Лучшая модель: RandomForest с настройкой гиперпараметров и отбором признаков")
print(f"F1-score (CV): 0.8536")  
print(f"Параметры: n_estimators=200, max_features=0.1, max_depth=15")
print(f"Количество отобранных признаков: {len(selected_indices)} из {len(feature_names)}")
print(f"Production Run ID: {production_run_id}")
print(f"Model Version: {version_number}")

Размер полной выборки: (302, 13)


d:\учеба\9 сем\mohov\.venv\Lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Используемые столбцы (13): ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


2025/10/14 21:57:47 INFO mlflow.tracking._tracking_service.client: 🏃 View run Production_Model_Full_Dataset at: http://127.0.0.1:5000/#/experiments/1/runs/128bd4a48b8e4f38a8a1c186094d85bf.
2025/10/14 21:57:47 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
Registered model 'heart_disease_model' already exists. Creating a new version of this model...
2025/10/14 21:57:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: heart_disease_model, version 4


Production Model Run ID: 128bd4a48b8e4f38a8a1c186094d85bf
Production модель зарегистрирована как версия 4 с тегом 'Production'
Production Run ID: 128bd4a48b8e4f38a8a1c186094d85bf

=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ ===
Лучшая модель: RandomForest с настройкой гиперпараметров и отбором признаков
F1-score (CV): 0.8536
Параметры: n_estimators=200, max_features=0.1, max_depth=15
Количество отобранных признаков: 10 из 25
Production Run ID: 128bd4a48b8e4f38a8a1c186094d85bf
Model Version: 4


Created version '4' of model 'heart_disease_model'.
